# NLP Lab 2 — Text Segmentation

Segmenting unspaced English text (e.g. `"itthatthecitytakestepstothisproblem"`) back into words, using two approaches:

1. **Greedy (Longest Match)** segmentation
2. **Dynamic Programming (Maximum Log-Probability)** segmentation

Evaluated using **Accuracy** (exact sentence match) and **Edit Distance** (character-level Levenshtein distance).

## 1. Load the Dataset

In [1]:
import json
import math

with open("text_segmentation_dataset.json", "r") as f:
    data = json.load(f)

word_counts = data["word_counts"]
test_cases = data["test_cases"]
total_corpus_words = data["metadata"]["total_corpus_words"]

vocab = set(word_counts.keys())
max_word_len = max(len(w) for w in vocab)

print(f"Vocabulary size       : {len(vocab)}")
print(f"Longest word length   : {max_word_len}")
print(f"Number of test cases  : {len(test_cases)}")

Vocabulary size       : 1500
Longest word length   : 14
Number of test cases  : 1000


## 2. Greedy (Longest Match) Segmentation

Starting from the left, repeatedly cut off the **longest** substring that exists in the vocabulary. If nothing matches at the current position, fall back to a single character so segmentation always makes progress.

In [2]:
def greedy_segment(text):
    words = []
    i = 0
    n = len(text)

    while i < n:
        matched = False
        max_len = min(max_word_len, n - i)
        for length in range(max_len, 0, -1):
            piece = text[i:i + length]
            if piece in vocab:
                words.append(piece)
                i += length
                matched = True
                break
        if not matched:
            words.append(text[i])
            i += 1

    return words

## 3. Dynamic Programming (Maximum Log-Probability) Segmentation

Builds up the best segmentation left to right, tracking the highest total log-probability path to every position. Word probability = `frequency / total_corpus_words`; unseen words get a small length-scaled penalty.

In [3]:
UNKNOWN_LOG_PROB = math.log(1e-8)

def word_log_prob(word):
    if word in word_counts:
        return math.log(word_counts[word] / total_corpus_words)
    return UNKNOWN_LOG_PROB * len(word)


def dp_segment(text):
    n = len(text)
    best_score = [-math.inf] * (n + 1)
    best_score[0] = 0.0
    back_pointer = [0] * (n + 1)

    for i in range(1, n + 1):
        max_len = min(max_word_len, i)
        for length in range(1, max_len + 1):
            j = i - length
            word = text[j:i]
            score = best_score[j] + word_log_prob(word)
            if score > best_score[i]:
                best_score[i] = score
                back_pointer[i] = j

    words = []
    i = n
    while i > 0:
        j = back_pointer[i]
        words.append(text[j:i])
        i = j
    words.reverse()
    return words

## 4. Evaluation Metrics

In [4]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],
                                    dp[i][j - 1],
                                    dp[i - 1][j - 1])
    return dp[m][n]


def evaluate(segment_function, name):
    correct = 0
    total_edit_dist = 0

    for case in test_cases:
        input_text = case["input"]
        ground_truth = case["ground_truth"]

        predicted_words = segment_function(input_text)
        predicted_sentence = " ".join(predicted_words)

        if predicted_sentence == ground_truth:
            correct += 1

        total_edit_dist += edit_distance(predicted_sentence, ground_truth)

    accuracy = correct / len(test_cases) * 100
    avg_edit_dist = total_edit_dist / len(test_cases)

    print(f"--- {name} ---")
    print(f"Accuracy      : {accuracy:.2f}%  ({correct}/{len(test_cases)} exact matches)")
    print(f"Edit Distance : {avg_edit_dist:.3f}  (average characters different from ground truth)")

    return accuracy, avg_edit_dist

## 5. Example Outputs

In [5]:
for case in test_cases[:3]:
    print(f"Input        : {case['input']}")
    print(f"Ground Truth : {case['ground_truth']}")
    print(f"Greedy       : {' '.join(greedy_segment(case['input']))}")
    print(f"DP           : {' '.join(dp_segment(case['input']))}")
    print()

Input        : itthatthecitytakestepstothisproblem
Ground Truth : it that the city take steps to this problem
Greedy       : it that the city takes t e p s to this problem
DP           : it that the city take steps to this problem

Input        : oftitlelawwasalsobythe
Ground Truth : of title law was also by the
Greedy       : of title law was also by the
DP           : of title law was also by the

Input        : failuretodothiswillcontinuetoplaceaon
Ground Truth : failure to do this will continue to place a on
Greedy       : failure to do this will continue top l a c e a on
DP           : failure to do this will continue to place a on



## 6. Full Evaluation

In [6]:
greedy_acc, greedy_ed = evaluate(greedy_segment, "Greedy (Longest Match)")
print()
dp_acc, dp_ed = evaluate(dp_segment, "Dynamic Programming (Max Log-Probability)")

--- Greedy (Longest Match) ---
Accuracy      : 69.10%  (691/1000 exact matches)
Edit Distance : 1.290  (average characters different from ground truth)



--- Dynamic Programming (Max Log-Probability) ---
Accuracy      : 98.20%  (982/1000 exact matches)
Edit Distance : 0.028  (average characters different from ground truth)


## 7. Summary Table

In [7]:
print(f"{'Method':35s}{'Accuracy':>12s}{'Edit Distance':>16s}")
print(f"{'Greedy (Longest Match)':35s}{greedy_acc:>11.2f}%{greedy_ed:>16.3f}")
print(f"{'Dynamic Programming':35s}{dp_acc:>11.2f}%{dp_ed:>16.3f}")

Method                                 Accuracy   Edit Distance
Greedy (Longest Match)                   69.10%           1.290
Dynamic Programming                      98.20%           0.028
